# Análise Horizontal da DRE — Empresas Listadas na B3

---

A análise horizontal é uma técnica de avaliação financeira que compara a evolução
de cada conta do demonstrativo ao longo do tempo, tomando um ano-base como referência.
O resultado expressa, em percentual, o quanto cada conta cresceu ou retraiu em relação
a esse ponto de partida.

Neste estudo, coletamos a DRE consolidada de uma empresa listada na B3 diretamente
da CVM, estruturamos os dados em formato de painel (contas × anos) e calculamos
a variação acumulada de cada conta em relação ao ano-base escolhido.

**Fonte dos dados:** http://dados.cvm.gov.br

## Dependências e Configurações Iniciais

---

Carregamento das bibliotecas e configuração do logger.

In [1]:
import tempfile
import requests
from   zipfile import ZipFile
from   pathlib import Path
from   loguru  import logger

import pandas as pd
import matplotlib.pyplot as plt

logger.info('BIBLIOTECAS CARREGADAS')

2026-06-11 16:50:12.334 | INFO     | __main__:<module>:10 - BIBLIOTECAS CARREGADAS


## Constantes e Configurações

---

In [2]:
DATA            = Path().cwd() / 'dados'
DATA.mkdir(parents=True, exist_ok=True)

URL_BASE        = 'https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/'
ANO_INICIO      = 2010
ANO_FIM         = 2025
DEMONSTRATIVO   = 'DRE_con'
ARQUIVOS_ZIP    = [f'dfp_cia_aberta_{ano}.zip' for ano in range(ANO_INICIO, ANO_FIM + 1)]

logger.info('CONSTANTES CARREGADAS')

2026-06-11 16:51:19.914 | INFO     | __main__:<module>:10 - CONSTANTES CARREGADAS


## 1. Coleta de Arquivos na CVM

---

Download e extração dos arquivos `.zip` anuais disponibilizados pela CVM.
O processo é idêntico ao utilizado no estudo de P/L histórico.

In [ ]:
logger.info('Iniciando download e extração de arquivos CSVs')
logger.info(f'Total de arquivos para download: {len(ARQUIVOS_ZIP)}')

with tempfile.TemporaryDirectory() as temp:
    temp_path = Path(temp)

    for arquivo in ARQUIVOS_ZIP:
        url_completa = URL_BASE + arquivo
        caminho_zip  = temp_path / arquivo

        try:
            logger.info(f'Baixando: {arquivo}')
            resposta = requests.get(url_completa, stream=True)
            resposta.raise_for_status()

            with open(caminho_zip, 'wb') as f:
                for chunk in resposta.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            logger.info(f'Extraindo {arquivo}...')
            with ZipFile(caminho_zip, 'r') as zip_ref:
                zip_ref.extractall(DATA)

        except requests.exceptions.HTTPError:
            logger.exception(f'Erro HTTP ao tentar baixar {arquivo}')
        except Exception:
            logger.exception(f'Erro genérico ao processar {arquivo}')

logger.info('Processo concluído com sucesso')